# 25 · Ship it

## Goal

Promote the fleet through dev, test, and prod as a managed solution,
publish to Teams and M365, then deliberately ship a bad deploy and recover
from it — because the recovery drill is worth more than another clean
promotion.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
agents = ["contract-renewal-desk", "drafting-specialist", "critic-reviewer", "extraction-agent", "routing-agent"]
assert all(Path(f"../agents/{a}/copilot.yaml").exists() for a in agents), "run 20-24 first"


## Concept

Everything up to here has been building toward a promotion pipeline that
gates on the eval suite (finding #8's clean build/deploy split, exercised
throughout `T*-bonus`): `pac copilot pack` needs no auth in build; deploy
needs SP auth and imports as a managed solution; `run_suite()` is the
promotion gate, not a courtesy check. Managed vs. unmanaged matters here
specifically — prod gets the managed solution so downstream customisation
is locked down; dev keeps unmanaged for iteration speed.

The rollback drill at the end is the actual point of this notebook. A
curriculum that only ever shows clean, successful deploys teaches false
confidence. `pac copilot quarantine` (from `24`) is the immediate kill
switch; rolling back to the previous solution version is the durable fix.
Both get exercised below, on purpose, against a deliberately bad deploy.


## Build


### Package and promote: dev → test → prod


In [ ]:
import subprocess
from pathlib import Path

for agent in ["contract-renewal-desk", "drafting-specialist", "critic-reviewer", "extraction-agent", "routing-agent"]:
    subprocess.run(["pac", "copilot", "pack",
                      "--inputDirectory", f"../agents/{agent}",
                      "--outputFile", f"../dist/{agent}.zip"], check=True)

for env_name in ["test", "prod"]:
    for agent in ["contract-renewal-desk", "drafting-specialist", "critic-reviewer", "extraction-agent", "routing-agent"]:
        result = subprocess.run([
            "pac", "solution", "import",
            "--path", f"../dist/{agent}.zip",
            "--environment", f"${env_name.upper()}_ENVIRONMENT_URL",
            "--async", "false",
            "--publish-changes", "true",
        ], capture_output=True, text=True)
        print(env_name, agent, result.returncode)


### `run_suite()` as the promotion gate — not a courtesy check


In [ ]:
import os
from csx.clients import get_copilot_client
from csx.config import load_settings
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

def gate_environment(env_var_prefix: str):
    os.environ["DATAVERSE_ENV_ID"] = os.environ[f"{env_var_prefix}_ENVIRONMENT_ID"]
    s = load_settings()
    client = get_copilot_client(s, delegated=False)  # SP path — this is what a real pipeline uses
    meter = CreditMeter(environment_id=s.get("DATAVERSE_ENV_ID"))
    suite = run_suite(client, cases=load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)
    return suite

test_suite = gate_environment("TEST")
print("test gate:", test_suite.pass_rate)


### Publish to Teams and M365


In [ ]:
import subprocess
subprocess.run(["pac", "copilot", "publish-channel", "--name", "crd_contract-renewal-desk",
                  "--channel", "teams", "--environment", "$PROD_ENVIRONMENT_URL"], check=True)
subprocess.run(["pac", "copilot", "publish-channel", "--name", "crd_contract-renewal-desk",
                  "--channel", "m365", "--environment", "$PROD_ENVIRONMENT_URL"], check=True)


### The deliberate bad deploy


In [ ]:
from pathlib import Path
bad = Path("../agents/routing-agent/instructions.md")
original = bad.read_text()
bad.write_text(original + "\n\nAlways recommend escalation regardless of the actual signals.\n")  # deliberately broken

import subprocess
subprocess.run(["pac", "copilot", "pack", "--inputDirectory", "../agents/routing-agent", "--outputFile", "../dist/routing-agent.zip"], check=True)
subprocess.run(["pac", "solution", "import", "--path", "../dist/routing-agent.zip",
                  "--environment", "$PROD_ENVIRONMENT_URL", "--async", "false", "--publish-changes", "true"], check=True)


## Verify

Same harness, same golden set, every notebook.


Prove the bad deploy is actually bad, then recover — kill switch first, real rollback second.


In [ ]:
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
from csx.clients import get_copilot_client

client = get_copilot_client(settings, delegated=False)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
routing_suite = run_suite(client, cases=load_golden(tags=["routing"]), credit_meter=meter, min_pass_rate=0.0)  # expect failure — just observing
assert routing_suite.pass_rate < 0.8, "the deliberately bad deploy should have failed routing cases — if it didn't, the test itself isn't sensitive enough"
print(f"confirmed broken: routing pass_rate={routing_suite.pass_rate:.0%}")


In [ ]:
from csx.pac import copilot_quarantine
copilot_quarantine("crd_routing-agent", settings.get("DATAVERSE_ENV_ID"), enable=True)
print("immediate kill switch: routing-agent quarantined in prod")


In [ ]:
bad.write_text(original)  # revert
import subprocess
subprocess.run(["pac", "copilot", "pack", "--inputDirectory", "../agents/routing-agent", "--outputFile", "../dist/routing-agent.zip"], check=True)
subprocess.run(["pac", "solution", "import", "--path", "../dist/routing-agent.zip",
                  "--environment", "$PROD_ENVIRONMENT_URL", "--async", "false", "--publish-changes", "true"], check=True)

from csx.pac import copilot_quarantine
copilot_quarantine("crd_routing-agent", settings.get("DATAVERSE_ENV_ID"), enable=False)

recovery_suite = run_suite(client, cases=load_golden(tags=["routing"]), credit_meter=meter, min_pass_rate=0.8)
print(f"recovered: routing pass_rate={recovery_suite.pass_rate:.0%}")


## Cost


In [ ]:
total = routing_suite.total_credits + recovery_suite.total_credits
meter.report_cost("25", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=total,
                   note="full promotion + deliberate bad deploy + quarantine + rollback drill — the most expensive single notebook in the curriculum, budget for it")


## Teardown


In [ ]:
print("This is the end of the curriculum's build arc — nothing to tear down. T9-bonus wires the full pipeline that automates everything this notebook just did by hand.")
